# Wrist Image Classification Pipeline
## 3-Class Classification: Vata, Pitta, Kapha

This notebook implements a complete PyTorch-based image classification pipeline using transfer learning with MobileNetV2.

## 1. Import Required Libraries

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load and Explore Dataset Structure

In [ ]:
# Define paths
DATA_ROOT = Path('.')
CLASSES = ['Vata', 'Pitta', 'Kapha']
TARGET_IMAGE = 'palm_open_back.jpg'

# Explore dataset structure
dataset_info = []

for class_name in CLASSES:
    class_path = DATA_ROOT / class_name
    if class_path.exists():
        subject_folders = [f for f in class_path.iterdir() if f.is_dir()]
        
        for subject_folder in subject_folders:
            image_path = subject_folder / TARGET_IMAGE
            metadata_path = subject_folder / 'metadata.json'
            
            if image_path.exists():
                # Extract gender from folder name
                folder_name = subject_folder.name
                gender = folder_name.split('_')[-1] if '_' in folder_name else 'unknown'
                
                # Load metadata if exists
                metadata = {}
                if metadata_path.exists():
                    with open(metadata_path, 'r') as f:
                        metadata = json.load(f)
                
                dataset_info.append({
                    'class': class_name,
                    'subject_id': folder_name,
                    'gender': gender,
                    'image_path': str(image_path),
                    'metadata': metadata
                })

# Create DataFrame
df = pd.DataFrame(dataset_info)
print(f"Total samples: {len(df)}")
print(f"\nClass distribution:")
print(df['class'].value_counts())
print(f"\nGender distribution:")
print(df['gender'].value_counts())
print(f"\nClass-Gender distribution:")
print(pd.crosstab(df['class'], df['gender']))

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
df['class'].value_counts().plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Gender distribution per class
pd.crosstab(df['class'], df['gender']).plot(kind='bar', ax=axes[1], stacked=False)
axes[1].set_title('Gender Distribution per Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Gender')

plt.tight_layout()
plt.show()

## 3. Define Data Preprocessing and Augmentation

In [ ]:
# ImageNet normalization values (for transfer learning)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transforms with augmentation
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=15),  # Small rotation for natural variation
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),  # Color augmentation
    transforms.RandomHorizontalFlip(p=0.5),  # Horizontal flip (anatomically valid for wrists)
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Validation/Test transforms (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("Data preprocessing and augmentation configured.")

## 4. Create Custom Dataset Class

In [ ]:
class WristDataset(Dataset):
    """Custom Dataset for wrist images."""
    
    def __init__(self, dataframe, transform=None):
        """
        Args:
            dataframe: DataFrame with 'image_path' and 'class' columns
            transform: torchvision transforms to apply
        """
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform
        
        # Map class names to indices
        self.class_to_idx = {cls: idx for idx, cls in enumerate(CLASSES)}
        self.idx_to_class = {idx: cls for cls, idx in self.class_to_idx.items()}
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get image path and label
        img_path = self.data.loc[idx, 'image_path']
        class_name = self.data.loc[idx, 'class']
        label = self.class_to_idx[class_name]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
    def get_class_name(self, idx):
        """Get class name from index."""
        return self.idx_to_class[idx]

print("WristDataset class defined.")

## 5. Split Data into Training and Validation Sets

In [ ]:
# Stratified split (80/20) to maintain class distribution
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['class'], 
    random_state=42
)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"\nTraining class distribution:")
print(train_df['class'].value_counts())
print(f"\nValidation class distribution:")
print(val_df['class'].value_counts())

# Create datasets
train_dataset = WristDataset(train_df, transform=train_transforms)
val_dataset = WristDataset(val_df, transform=val_transforms)

# Create dataloaders
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nDataLoaders created with batch size: {BATCH_SIZE}")

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
fig.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')

for i, class_name in enumerate(CLASSES):
    class_samples = train_df[train_df['class'] == class_name].sample(n=4, random_state=42)
    
    for j, (idx, row) in enumerate(class_samples.iterrows()):
        img = Image.open(row['image_path'])
        axes[i, j].imshow(img)
        axes[i, j].set_title(f"{class_name}\n{row['subject_id']}", fontsize=10)
        axes[i, j].axis('off')

plt.tight_layout()
plt.show()

## 6. Build Model Architecture with Transfer Learning

In [ ]:
class WristClassifier(nn.Module):
    """MobileNetV2-based classifier for wrist images."""
    
    def __init__(self, num_classes=3, pretrained=True):
        super(WristClassifier, self).__init__()
        
        # Load pre-trained MobileNetV2
        self.backbone = models.mobilenet_v2(pretrained=pretrained)
        
        # Get the number of input features for the classifier
        in_features = self.backbone.classifier[1].in_features
        
        # Replace the classifier with custom head
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

# Initialize model
model = WristClassifier(num_classes=len(CLASSES), pretrained=True)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: MobileNetV2-based WristClassifier")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model moved to: {device}")

## 7. Define Loss Function and Optimizer

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer (Adam with learning rate 1e-4)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Learning rate scheduler (reduce on plateau)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max',  # Monitor validation accuracy
    factor=0.5, 
    patience=5
)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam (lr=1e-4, weight_decay=1e-5)")
print("Scheduler: ReduceLROnPlateau")

## 8. Define Training and Validation Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1


def validate_epoch(model, dataloader, criterion, device):
    """Validate for one epoch."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Track metrics
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels

print("Training and validation functions defined.")

## 9. Train the Model with Early Stopping

In [ ]:
# Training configuration
NUM_EPOCHS = 50
PATIENCE = 10  # Early stopping patience
SAVE_PATH = './model.pth'

# Training history
history = {
    'train_loss': [], 'train_acc': [], 'train_f1': [],
    'val_loss': [], 'val_acc': [], 'val_f1': []
}

# Early stopping variables
best_val_acc = 0.0
patience_counter = 0
best_model_state = None

print("Starting training...")
print(f"Epochs: {NUM_EPOCHS}, Early stopping patience: {PATIENCE}\n")

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc, train_f1 = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validate
    val_loss, val_acc, val_f1, _, _ = validate_epoch(
        model, val_loader, criterion, device
    )
    
    # Update history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    # Learning rate scheduler step
    scheduler.step(val_acc)
    
    # Print progress
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    
    # Early stopping check
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        print(f"  ✓ Best model updated (Val Acc: {best_val_acc:.4f})")
    else:
        patience_counter += 1
        print(f"  Patience: {patience_counter}/{PATIENCE}")
    
    print()
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(best_model_state)
print(f"\nTraining completed. Best validation accuracy: {best_val_acc:.4f}")

## 10. Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], 'b-', label='Train Accuracy', linewidth=2)
axes[1].plot(epochs_range, history['val_acc'], 'r-', label='Val Accuracy', linewidth=2)
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(epochs_range, history['train_f1'], 'b-', label='Train F1', linewidth=2)
axes[2].plot(epochs_range, history['val_f1'], 'r-', label='Val F1', linewidth=2)
axes[2].set_title('Training and Validation F1 Score', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Evaluate Model Performance

In [ ]:
# Get validation predictions with best model
val_loss, val_acc, val_f1, val_preds, val_labels = validate_epoch(
    model, val_loader, criterion, device
)

print("=" * 60)
print("FINAL VALIDATION RESULTS")
print("=" * 60)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Validation F1 Score: {val_f1:.4f}")
print()

# Classification report
print("Classification Report:")
print(classification_report(
    val_labels, 
    val_preds, 
    target_names=CLASSES,
    digits=4
))

# Confusion matrix
cm = confusion_matrix(val_labels, val_preds)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix - Validation Set', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-Class Accuracy:")
for i, class_name in enumerate(CLASSES):
    class_mask = np.array(val_labels) == i
    class_acc = accuracy_score(
        np.array(val_labels)[class_mask],
        np.array(val_preds)[class_mask]
    )
    print(f"  {class_name}: {class_acc:.4f}")

## 12. Save the Trained Model

In [ ]:
# Save model checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_to_idx': train_dataset.class_to_idx,
    'idx_to_class': train_dataset.idx_to_class,
    'model_architecture': 'MobileNetV2',
    'num_classes': len(CLASSES),
    'input_size': (224, 224),
    'best_val_acc': best_val_acc,
    'best_val_f1': val_f1,
    'normalization': {
        'mean': IMAGENET_MEAN,
        'std': IMAGENET_STD
    }
}

torch.save(checkpoint, SAVE_PATH)
print(f"Model saved to: {SAVE_PATH}")
print(f"Model size: {os.path.getsize(SAVE_PATH) / (1024 * 1024):.2f} MB")
print("\nCheckpoint contents:")
for key in checkpoint.keys():
    if key != 'model_state_dict':
        print(f"  - {key}: {checkpoint[key]}")

## 13. Test Inference Function

In [ ]:
def predict(image_path, model_path=SAVE_PATH):
    """
    Predict class for a single wrist image.
    
    Args:
        image_path: Path to the image file
        model_path: Path to the saved model checkpoint
        
    Returns:
        Dictionary with prediction results
    """
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    
    # Initialize model
    model = WristClassifier(num_classes=checkpoint['num_classes'], pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    # Prepare image
    image = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=checkpoint['normalization']['mean'],
            std=checkpoint['normalization']['std']
        )
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)
        predicted_class = checkpoint['idx_to_class'][predicted_idx.item()]
    
    # Get all class probabilities
    class_probs = {
        checkpoint['idx_to_class'][i]: probabilities[0][i].item()
        for i in range(len(checkpoint['idx_to_class']))
    }
    
    return {
        'predicted_class': predicted_class,
        'confidence': confidence.item(),
        'probabilities': class_probs
    }

# Test on a random validation image
test_sample = val_df.sample(n=1, random_state=42).iloc[0]
test_image_path = test_sample['image_path']
true_class = test_sample['class']

result = predict(test_image_path)

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)
print(f"Test image: {test_sample['subject_id']}")
print(f"True class: {true_class}")
print(f"\nPredicted class: {result['predicted_class']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"\nAll class probabilities:")
for cls, prob in result['probabilities'].items():
    print(f"  {cls}: {prob:.4f}")

# Display the test image
plt.figure(figsize=(8, 8))
img = Image.open(test_image_path)
plt.imshow(img)
plt.title(f"True: {true_class} | Predicted: {result['predicted_class']} ({result['confidence']:.2%})", 
          fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()